# 03: Route Overlap & Network Coverage
**Milestone 2: Week 3 — Step 6**

Using the Spring timetable (`2026-04-09`), this notebook evaluates corridor-level redundancy vs planned coordination:
- **Overlapping Pairs Definition:** Any pair sharing at least one stop ($|S_A \cap S_B| \ge 1$). 127 pairs meet this criterion out of 820 possible Manhattan pairs.
- **Classification Rule:**
  - **Same route family (`planned`):** Pairs with the same route number, such as a local route and its Select Bus version, or lettered variants (`M15/M15+`, `M34+/M34A+`, `M14A+/M14D+`).
  - **Shared corridor (`review`):** Different route numbers that share stops (`M101/M103`, `M101/M102`, `M1/M2`, etc.).
- **Network coverage metric:** Share of stops served by no other route (stop exclusivity).
- **Interactive Folium map:** Visualizes top overlapping corridors.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.eda_utils import typical_days, DAY_TYPE_COLORS, calc_period_ratio, calc_cv

DATA = ROOT / "data" / "processed"
FIG = ROOT / "reports" / "figures"
TAB = ROOT / "reports" / "tables"
FIG.mkdir(parents=True, exist_ok=True)
TAB.mkdir(parents=True, exist_ok=True)

matched = pd.read_parquet(DATA / "matched_hourly.parquet")
profiles = pd.read_parquet(DATA / "route_profiles.parquet")
print(f"Loaded matched hourly: {len(matched):,} rows (41 routes)")
print(f"Loaded route profiles: {len(profiles)} routes")


Loaded matched hourly: 233,227 rows (41 routes)
Loaded route profiles: 41 routes


## Step 6.1: Pairwise Route Overlap (Spring 2026-04-09 Timetable)
Analyzes stop-level overlap across all 127 route pairs that share at least one stop ($|S_A \cap S_B| \ge 1$).

In [2]:
pairs_df = pd.read_csv(TAB / 'm2_route_overlap_all_pairs.csv')
print(f'Total overlapping route pairs (>= 1 shared stop): {len(pairs_df)}')
print('\nTop 10 Route Pairs by Overlap Share:')
pairs_df.head(10)

Total overlapping route pairs (>= 1 shared stop): 127

Top 10 Route Pairs by Overlap Share:


,route_1,route_2,smaller_route,larger_route,shared_stops_count,smaller_route_stops,larger_route_stops,overlap_share,jaccard_similarity
0,M101,M103,M103,M101,82,106,155,0.7736,0.4581
1,M34+,M34A+,M34A+,M34+,22,29,34,0.7586,0.5366
2,M102,M103,M103,M102,77,106,114,0.7264,0.5385
3,M101,M102,M102,M101,76,114,155,0.6667,0.3938
4,M1,M2,M1,M2,77,134,144,0.5746,0.3831
5,M2,M3,M2,M3,79,144,162,0.5486,0.3480
6,M31,M57,M57,M31,23,42,57,0.5476,0.3026
7,M106,M96,M96,M106,12,24,33,0.5000,0.2667
8,M1,M3,M1,M3,64,134,162,0.4776,0.2759
9,M14A+,M14D+,M14A+,M14D+,19,42,44,0.4524,0.2836


## Step 6.2: Top 15 Overlapping Pairs In-Depth Analysis
Profiles corridor continuity, direction alignment, operating hours, peak frequency, and design classification ('planned' vs 'review').

In [3]:
top15 = pd.read_csv(TAB / 'm2_top_15_route_overlaps.csv')
top15[['rank', 'route_1', 'route_2', 'shared_stops', 'overlap_share', 'jaccard', 'stretch_type', 'combined_peak_buses_per_hr', 'design_classification', 'design_notes']]

,rank,route_1,route_2,shared_stops,overlap_share,jaccard,stretch_type,combined_peak_buses_per_hr,design_classification,design_notes
0,1,M101,M103,82,77.4%,45.8%,Continuous Corridor,66.7,review,Shared corridor (different route numbers shari...
1,2,M34+,M34A+,22,75.9%,53.7%,Continuous Corridor,31.7,planned,Same route family (same route number or letter...
2,3,M102,M103,77,72.6%,53.8%,Continuous Corridor,49.5,review,Shared corridor (different route numbers shari...
3,4,M101,M102,76,66.7%,39.4%,Segmented / Multi-Stretch,70.8,review,Shared corridor (different route numbers shari...
4,5,M1,M2,77,57.5%,38.3%,Continuous Corridor,64.0,review,Shared corridor (different route numbers shari...
5,6,M2,M3,79,54.9%,34.8%,Segmented / Multi-Stretch,60.0,review,Shared corridor (different route numbers shari...
6,7,M31,M57,23,54.8%,30.3%,Continuous Corridor,41.5,review,Shared corridor (different route numbers shari...
7,8,M106,M96,12,50.0%,26.7%,Continuous Corridor,49.7,review,Shared corridor (different route numbers shari...
8,9,M1,M3,64,47.8%,27.6%,Segmented / Multi-Stretch,65.7,review,Shared corridor (different route numbers shari...
9,10,M14A+,M14D+,19,45.2%,28.4%,Continuous Corridor,61.5,planned,Same route family (same route number or letter...


## Step 6.3: Network Coverage & Exclusive Stop Share
Measures the share of stops served by no other route. M35 is the only route in the Manhattan timetable serving any of its 23 stops.

In [4]:
coverage = pd.read_csv(TAB / 'm2_route_stop_coverage.csv')
print('Top 10 Routes with Highest Stop Exclusivity:')
print(coverage.head(10)[['route', 'total_stops', 'exclusive_stops', 'exclusive_stop_share']])

print('\nTop 5 Routes with Lowest Stop Exclusivity (Parallel Corridors):')
print(coverage.tail(5)[['route', 'total_stops', 'exclusive_stops', 'exclusive_stop_share']])

Top 10 Routes with Highest Stop Exclusivity:
  route  total_stops  exclusive_stops  exclusive_stop_share
0   M35           23               23                1.0000
1   M42           32               30                0.9375
2  M79+           27               24                0.8889
3   M10           88               78                0.8864
4    M8           42               36                0.8571
5   M50           32               27                0.8438
6  M23+           31               26                0.8387
7  M60+           35               29                0.8286
8   M15          122              100                0.8197
9   M66           26               21                0.8077

Top 5 Routes with Lowest Stop Exclusivity (Parallel Corridors):
    route  total_stops  exclusive_stops  exclusive_stop_share
36     M4          152               32                0.2105
37   M103          106               20                0.1887
38  M34A+           29                4     

## Step 6.4: Interactive Corridor Map (Folium)
Visualizes top overlapping transit corridors on OpenStreetMap.

In [5]:
import folium
from IPython.display import IFrame

map_file = FIG / 'route_overlaps.html'
print(f'Interactive map generated at: {map_file.relative_to(ROOT)}')
# Render iframe preview
IFrame(src='../reports/figures/route_overlaps.html', width='100%', height=500)

Interactive map generated at: reports\figures\route_overlaps.html
